# Process Multiple Long Strips

In [1]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

import papermill

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [3]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
0,NaN,2025-03-26,GHL_pyapp_20250326T1258,oven aging test day 1,strip 7,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2025-03-26,GHL_pyapp_20250326T1403,oven aging test day 1,strip 10,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-03-26,GHL_pyapp_20250326T1416,oven aging test day 1,strip 14,NaN,"old, with tape",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2025-03-27,GHL_pyapp_20250327T1428,oven aging test day 2,strip 7,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,2025-03-27,GHL_pyapp_20250327T1438,oven aging test day 2,strip 10,NaN,"old, with 3d print clamps",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,140.0,2025-07-11,GHL_WaxRobot_20250711T1438,July11WaxrobotStrips,nomarks,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,"light level in ""red"", improves snr?"
146,141.0,2025-07-11,GHL_WaxRobot_20250711T1442,July11WaxrobotStrips,nomarks,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,"light level in ""green"""
147,142.0,2025-07-11,GHL_WaxRobot_20250711T1501,July11WaxrobotStrips,"1.0b,140c,s:50",NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,cut strip to the wax strip
148,143.0,2025-07-11,GHL_WaxRobot_20250711T1513,July11WaxrobotStrips,"1.2b,130c,s:30",NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,cut strip to the wax strip


In [4]:
#%% Filter The Tests To Process
dfmasks = [
    #(dftests['Test date']=='2025-06-11') | (dftests['Test date']=='2025-06-12'),
    
    dftests['Test date']=='2025-07-11',
    
    #dftests['Test ID'].isin([47,65]) # strips flagged as bad from May batch

    # strips flagged as "extra bad" from June 09 Batch
    #dftests['Test ID'].isin([73,77,85,88,97,117])
    #dftests['Test ID'].isin([97]),

    #dftests['Test name']!='GHL_pyapp_20250512T',
    #dftests['ProcessingNotes'].str.startswith('done,2'),
    #~dftests['Batch'].str.contains('Valve')
]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

,Test ID,Test date,Test name,Batch,Strip,Wax (mg/mm),Fixture,"X, FOV","Y, FOV","Z, FOV",...,Right edge (mm),Left edge (mm),Strip length (mm),Speed/Sensitivity,Averaging (A-scan),Refractive Index,File size (MB),Total size (GB),ProcessingNotes,Notes
145,140.0,2025-07-11,GHL_WaxRobot_20250711T1438,July11WaxrobotStrips,nomarks,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,"light level in ""red"", improves snr?"
146,141.0,2025-07-11,GHL_WaxRobot_20250711T1442,July11WaxrobotStrips,nomarks,NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,"light level in ""green"""
147,142.0,2025-07-11,GHL_WaxRobot_20250711T1501,July11WaxrobotStrips,"1.0b,140c,s:50",NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,cut strip to the wax strip
148,143.0,2025-07-11,GHL_WaxRobot_20250711T1513,July11WaxrobotStrips,"1.2b,130c,s:30",NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fail on 0615,cut strip to the wax strip
149,144.0,2025-07-11,GHL_WaxRobot_20250711T1519,July11WaxrobotStrips,"1.2b,130c,s:30",NaN,Vacuum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success on 0615,"cut strip to the wax strip; light level in ""re..."


In [5]:
#%% Load OCT study information
octstudies = [];
folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
#folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


STUDY: GHL_WaxRobot_20250711T1438
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 22,
    'study_num_vtk_files': 0}
STUDY: GHL_WaxRobot_20250711T1442
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 22,
    'study_num_vtk_files': 0}
STUDY: GHL_WaxRobot_20250711T1501
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 29,
    'study_num_vtk_files': 0}
STUDY: GHL_WaxRobot_20250711T1513
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 30,
    'study_num_vtk_files': 0}
STUDY: GHL_WaxRobot_20250711T1519
{   'study_has_an_ini_file': True,
    'study_

In [6]:
# Master Parameters
oct_scalar_min = 30;
oct_scalar_max = 60;

In [7]:
#%% Load OCTSTUDY object, and start some processing on it
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print(octstudy)

    fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    
    if(octstudy.study_info['study_num_oct_files']>0):
        # --Loading And Pre-Processing--
        if(fname_merged_and_rescaled_volume.exists()):
            pass;
            #print(f'Loading {fname_merged_and_rescaled_volume.name}')
            # Load the strip and merge into one volume
            #vdvol = vedo.Volume(pv.read(fname_merged_and_rescaled_volume));
            #octstudy.vdvol = vdvol;
        else:
            # Load OCT Data for this study
            octstudy.load_all_octs();


            # THESE WILL DO NOTHING IF ANTICIPATED OUTPUTS/ARTIFACTS ALREADY EXIST IN THE PROCESSED FOLDER

            # RGB Camera Images - Write them out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

            # RGB Camera Images - Make a montage and write out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_montage_image(octstudy);
            #break;


            # IF MERGED STACKED AND RESCALED TO SCALAR RANGE FILE EXISTS, LOAD THAT; OTHERWISE PROCESS IT HERE
            octstudy.folder_study_processed.mkdir(exist_ok=True);
            fname = fname_merged_and_rescaled_volume;
            if(fname.exists()):
                print('Stacked volume byte-size .vtk file already exists, will not recreate.');
                print(fname);
            else:
                print(f'Generating merged and rescaled .vtk volume');
                # Generate merged and rescaled volume
                vdvol = octstudy.generate_merged_vdvol_and_rescaled(oct_scalar_min,oct_scalar_max);
                octstudy.vdvol = vdvol;
            
                # save this byte-adjusted volume
                vdvol.dataset.save(fname);
            
            # Unload OCT Data for this study (we will still keep the vdvol)
            octstudy.unload_all_octdata();

        # --Detailed Image Processing--
        if hasattr(octstudy,'vdvol'):
            del octstudy.vdvol;

<OCT_Study_Folder Object>
GHL_WaxRobot_20250711T1438 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 22,
    'study_num_vtk_files': 0}
<OCT_Study_Folder Object>
GHL_WaxRobot_20250711T1442 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 22,
    'study_num_vtk_files': 0}
<OCT_Study_Folder Object>
GHL_WaxRobot_20250711T1501 in folder //file.corp.ghlabs.org/Shared/Projects/NAATOS/V1/NAATOS_OCT_WORK/OCTExport
{   'study_has_an_ini_file': True,
    'study_has_json_info_file': True,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 29,
    'study_num_vtk_files': 0

# (util) delete some items from processed folder

In [8]:
# Warning this can be destructive deleting processed data!!
if False:
    for idx,octstudy in enumerate(octstudies):
        print('~~~~~~~');
        print(octstudy.name);
        
        if False:
            fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
            if(fname.exists()):
                print('deleted',fname.name);
                fname.unlink();
            
            fname = (octstudy.folder_study_processed/'data_extracted.npz')
            if(fname.exists()):
                print('deleted',fname.name);
                fname.unlink();

            flist = octstudy.folder_study_processed.glob('figout*');
            for fname in flist:
                print('deleted',fname.name);
                fname.unlink();

            flist = octstudy.folder_study_processed.glob('processing_step*');
            for fname in flist:
                print('deleted',fname.name);
                fname.unlink();

        if True:
            # move files to a backup folder in the processed directory

            folder_backup = (octstudy.folder_study_processed/'_backup20250708');
            folder_backup.mkdir(exist_ok=True);

            fname = (octstudy.folder_study_processed/'along_strip_data_extracted.hdf5')
            if(fname.exists()):
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);
        
            fname = (octstudy.folder_study_processed/'data_extracted.npz')
            if(fname.exists()):
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);

            flist = octstudy.folder_study_processed.glob('figout*');
            for fname in flist:
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);

            flist = octstudy.folder_study_processed.glob('processing_step*');
            for fname in flist:
                print('move to backup',fname.name);
                fname.rename(folder_backup/fname.name);
        #break;


# Call Notebooks - Processing A to D and E

In [9]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
norunlist = [
    #'GHL_pyapp_20250508T1112','GHL_pyapp_20250508T1128','GHL_pyapp_20250508T1134','GHL_pyapp_20250508T1143'
]
forcerunlist = [
    #'GHL_pyapp_20250508T1151',
    #'GHL_pyapp_20250508T1601',
    #'GHL_pyapp_20250508T1608',
]
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    rescheck = octstudy.resultsCheck();
    pp.pprint(rescheck)
    doWeRunTheNotebook = any([v is False for k,v in rescheck.items()])
    #doWeRunTheNotebook = doWeRunTheNotebook or ('/dfstepE' not in rescheck['along_strip_data_extracted']);
    doWeRunTheNotebook = doWeRunTheNotebook or ('/dfstepA' not in rescheck['along_strip_data_extracted']);
    print('Run?',doWeRunTheNotebook)
    if((doWeRunTheNotebook and octstudy.name not in norunlist) or (octstudy.name in forcerunlist)):
        octstudies_to_run.append(octstudy);
    # if(octstudy.name in forcerunlist):
    #     octstudies_to_run.append(octstudy);

# if(len(octstudies_to_run)>4):
#     #octstudies_to_run=octstudies_to_run[0:4];
#     #octstudies_to_run=octstudies_to_run[-4:];
#     print('only a subset');
#     norunlist = [
#         'GHL_pyapp_20250508T1112','GHL_pyapp_20250508T1128','GHL_pyapp_20250508T1134','GHL_pyapp_20250508T1143'
#     ]
#     octstudies_to_run = [s for s in octstudies_to_run if s.name not in norunlist ]
#     pass;

#octstudies_to_run = octstudies_to_run[0:20];

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

~~~~~~~
GHL_WaxRobot_20250711T1438
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False}
Run? True
~~~~~~~
GHL_WaxRobot_20250711T1442
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False}
Run? True
~~~~~~~
GHL_WaxRobot_20250711T1501
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False}
Run? True
~~~~~~~
GHL_WaxRobot_20250711T1513
{   'along_strip_data_extracted': False,
    'exist_data_extracted': False,
    'figoutmp4_stepA': False}
Run? True
~~~~~~~
GHL_WaxRobot_20250711T1519
{   'along_strip_data_extracted': ['/dfstepA'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True}
Run? False
~~~~~~~
We will run the processing on 4 octstudies.
['GHL_WaxRobot_20250711T1438', 'GHL_WaxRobot_20250711T1442', 'GHL_WaxRobot_20250711T1501', 'GHL_WaxRobot_20250711T1513']


# Prepare notebooks we will call

In [10]:
nbpaths = [
    #r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250615_process_a_longstrip.ipynb",
    r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250721_process_a_longstrip.ipynb",
];
for nbpath in nbpaths:
    print('Notebook:',Path(nbpath).name);
    parameters = papermill.inspect_notebook(nbpath)
    pp.pprint(parameters.keys());

Notebook: octproc_20250721_process_a_longstrip.ipynb
dict_keys(['oct_scalar_min', 'oct_scalar_max', 'codename', 'study_name', 'folder_octexport_root', 'folder_figure_temp'])


# Call notebooks to process (Multiple-strips in parallel, concurrent futures)

In [11]:
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):

    folder_temp_path = Path(r'D:\TEMP\OCTtmp');
    folder_temp_path.mkdir(parents=True,exist_ok=True);

    # run each notebook we have defined to be run
    nnotebooks = len(nbpaths);
    for count,nbpath in enumerate(nbpaths):
        nbpath = Path(nbpath)
        nbpath_out = folder_temp_path/(nbpath.stem+'_OUT_{:s}.ipynb').format(octstudy.name)

        # execute a notebook
        print(f'Launching for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} ...')
        try:
            papermill.execute_notebook(
                input_path=nbpath,
                output_path=nbpath_out,
                parameters=dict(
                    folder_octexport_root=folder_octexport_root.as_posix(),
                    codename = Path(nbpath).name,
                    study_name=octstudy.name,
                    folder_figure_temp=nbpath_out.parent.as_posix()
                )
            )
            print(f'Finished for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} !')
        except Exception as e:
            print(f'FAILED for {octstudy.name} the {count+1}/{nnotebooks} notebook {nbpath.name} !')
            break;
    print(f'alldone for {octstudy.name}')


# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result());

# Example usage
num_processes = 6;  # Number of parallel processes
run_in_parallel(num_processes);

Launching for GHL_WaxRobot_20250711T1438 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb ...
Launching for GHL_WaxRobot_20250711T1442 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb ...
Launching for GHL_WaxRobot_20250711T1501 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb ...
Launching for GHL_WaxRobot_20250711T1513 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb ...


Executing:   0%|          | 0/39 [00:00<?, ?cell/s]

Executing:   0%|          | 0/39 [00:00<?, ?cell/s]

Executing:   0%|          | 0/39 [00:00<?, ?cell/s]

Executing:   0%|          | 0/39 [00:00<?, ?cell/s]

Autosave too slow: 15.57 sec, over 25% limit. Backing off to 60 sec
Autosave too slow: 19.91 sec, over 25% limit. Backing off to 60 sec
Autosave too slow: 17.06 sec, over 25% limit. Backing off to 60 sec
Autosave too slow: 9.57 sec, over 25% limit. Backing off to 60 sec
Autosave too slow: 23.39 sec, over 25% limit. Backing off to 120 sec
Autosave too slow: 24.18 sec, over 25% limit. Backing off to 120 sec
Autosave too slow: 27.18 sec, over 25% limit. Backing off to 120 sec


Finished for GHL_WaxRobot_20250711T1438 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb !
alldone for GHL_WaxRobot_20250711T1438
Finished for GHL_WaxRobot_20250711T1501 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb !
alldone for GHL_WaxRobot_20250711T1501
Finished for GHL_WaxRobot_20250711T1442 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb !
alldone for GHL_WaxRobot_20250711T1442
Finished for GHL_WaxRobot_20250711T1513 the 1/1 notebook octproc_20250721_process_a_longstrip.ipynb !
alldone for GHL_WaxRobot_20250711T1513


KeyboardInterrupt: 